## 🔐 Prerequisites

Before running the first cell, make sure you're authenticated with Azure CLI:

```bash
az login
```

# 🏦 Suspend and Resume Conversation Threads

## Industry Use Case: Insurance Claim Processing Continuity

This notebook demonstrates how to **suspend and resume conversation threads** comparing service-managed (Azure AI) and in-memory threading approaches.

| Feature | FSI Application |
|---------|----------------|
| **Service-Managed Threads** | Azure-hosted claim conversations |
| **In-Memory Threads** | Local processing with custom storage |
| **Session Persistence** | Resume claims across sessions |
| **Multi-Device Access** | Continue claims from any device |

In [5]:
import os
from dotenv import load_dotenv

load_dotenv('../../.env', override=True)

print(f"✅ Environment loaded")
print(f"Project Endpoint: {os.getenv('AI_FOUNDRY_PROJECT_ENDPOINT')[:50]}...")

✅ Environment loaded
Project Endpoint: https://demopocaifoundry.services.ai.azure.com/api...


In [6]:
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print("✅ All imports loaded")

✅ All imports loaded


## Define Insurance Claim Tools

In [7]:
async def get_claim_status(claim_id: str) -> str:
    """Get insurance claim status."""
    claims = {
        "CLM-001": {"status": "Under Review", "type": "Auto", "amount": 5200.00, "filed": "2026-01-15"},
        "CLM-002": {"status": "Approved", "type": "Home", "amount": 15000.00, "filed": "2026-01-10"},
        "CLM-003": {"status": "Pending Documents", "type": "Health", "amount": 3500.00, "filed": "2026-01-17"},
    }
    c = claims.get(claim_id, {"status": "Not Found", "type": "Unknown", "amount": 0, "filed": "N/A"})
    return f"Claim {claim_id}: {c['type']} claim for ${c['amount']:,.2f}, Status: {c['status']}, Filed: {c['filed']}"


async def submit_additional_documents(claim_id: str, document_type: str) -> str:
    """Submit additional documents for a claim."""
    return f"Document '{document_type}' submission initiated for {claim_id}. Upload link will be sent to your registered email."


print("✅ Tools defined: get_claim_status, submit_additional_documents")

✅ Tools defined: get_claim_status, submit_additional_documents


## Example 1: Thread Suspend and Resume

With `FoundryChatClient`, threads manage conversation history locally. Serialization captures the full message state, enabling complete conversation restoration.

> **Note:** Previously, the Azure AI Agent service managed threads server-side with lightweight serialization (~50 bytes). With `FoundryChatClient`, thread state is managed locally and serialization includes the full message history.

In [8]:
async def suspend_resume_thread_example1():
    """Suspend and resume a session using FoundryChatClient."""
    print("\n--- Suspend-Resume Thread (Example 1) ---\n")
    
    PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
    
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=AzureCliCredential(),
    )
    agent = Agent(
        client=client,
        name="ClaimsAdvisor",
        instructions="You are an insurance claims advisor. Help customers with their claim inquiries.",
        tools=[get_claim_status, submit_additional_documents],
    )
    
    # Start new session
    session = agent.create_session()
    print("✅ Session created\n")
    
    # Phase 1: Initial conversation
    query1 = "Hi, can you check the status of my claim CLM-003?"
    print(f"Customer: {query1}")
    response1 = await agent.run(query1, session=session)
    print(f"Advisor: {response1.text}\n")
    
    # Phase 2: Serialize (suspend)
    serialized_session = session.to_dict()
    print(f"💾 Serialized session: {serialized_session}")
    print(f"📊 Size: ~{len(str(serialized_session))} bytes\n")
    
    # Phase 3: Deserialize (resume)
    from agent_framework import AgentSession
    resumed_session = AgentSession.from_dict(serialized_session)
    print("✅ Session deserialized\n")
    
    query2 = "What documents do I need to provide?"
    print(f"Customer: {query2}")
    response2 = await agent.run(query2, session=resumed_session)
    print(f"Advisor: {response2.text}\n")
    
    print("✅ Thread suspend/resume demo completed!\n")

await suspend_resume_thread_example1()


--- Suspend-Resume Thread (Example 1) ---

✅ Session created

Customer: Hi, can you check the status of my claim CLM-003?
Advisor: The status of your claim (CLM-003) for $3,500.00 is currently listed as **Pending Documents**. It was filed on **January 17, 2026**. You will need to submit the required documents to proceed with the claim. Please let me know if you’d like assistance with this!

💾 Serialized session: {'type': 'session', 'session_id': '04566825-7208-4469-aef9-f8053d274f5c', 'service_session_id': 'resp_00f87f3a2790d58f0069cda4cd8c808193b0999aa1603d6c11', 'state': {}}
📊 Size: ~167 bytes

✅ Session deserialized

Customer: What documents do I need to provide?
Advisor: For claim CLM-003, you need to provide the following documents, as they are commonly required for health claims pending documentation:

1. **Medical Reports**: Detailed reports from your healthcare provider outlining the diagnosis and treatment.
2. **Hospital Bills**: Itemized bills showing services rendered and c

## Example 2: Thread Suspend and Resume (Second Scenario)

This example demonstrates the same suspend/resume pattern with a different claim inquiry, showing that the pattern works consistently across conversations.

In [9]:
async def suspend_resume_thread_example2():
    """Suspend and resume a session - second scenario."""
    print("\n--- Suspend-Resume Thread (Example 2) ---\n")
    
    PROJECT_ENDPOINT = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
    MODEL_DEPLOYMENT = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-4o")
    
    client = FoundryChatClient(
        project_endpoint=PROJECT_ENDPOINT,
        model=MODEL_DEPLOYMENT,
        credential=AzureCliCredential(),
    )
    agent = Agent(
        client=client,
        name="ClaimsAdvisor",
        instructions="You are an insurance claims advisor. Help customers with their claim inquiries.",
        tools=[get_claim_status, submit_additional_documents],
    )
    
    # Start new session
    session = agent.create_session()
    print("✅ Session created\n")
    
    # Phase 1: Initial conversation
    query1 = "Check status of claim CLM-001 please"
    print(f"Customer: {query1}")
    response1 = await agent.run(query1, session=session)
    print(f"Advisor: {response1.text}\n")
    
    # Phase 2: Serialize (suspend)
    serialized_session = session.to_dict()
    print(f"💾 Serialized session (first 150 chars): {str(serialized_session)[:150]}...")
    print(f"📊 Size: ~{len(str(serialized_session))} bytes (full message history)\n")
    
    # Phase 3: Deserialize (resume)
    from agent_framework import AgentSession
    resumed_session = AgentSession.from_dict(serialized_session)
    print("✅ Session deserialized with full history\n")
    
    query2 = "What was the claim amount again?"
    print(f"Customer: {query2}")
    response2 = await agent.run(query2, session=resumed_session)
    print(f"Advisor: {response2.text}\n")
    
    print("✅ Thread suspend/resume demo completed!\n")

await suspend_resume_thread_example2()


--- Suspend-Resume Thread (Example 2) ---

✅ Session created

Customer: Check status of claim CLM-001 please
Advisor: The status of claim CLM-001 is as follows:

- **Type**: Auto claim
- **Amount**: $5,200.00
- **Status**: Under Review
- **Filed on**: January 15, 2026

If you need further assistance or want to submit additional documents, let me know!

💾 Serialized session (first 150 chars): {'type': 'session', 'session_id': '1c172776-9326-423f-8254-75db4636df4b', 'service_session_id': 'resp_08d49e63d6cdfdff0069cda4e92abc8195a7f3b2e664a929...
📊 Size: ~167 bytes (full message history)

✅ Session deserialized with full history

Customer: What was the claim amount again?
Advisor: The claim amount for CLM-001 is **$5,200.00**.

✅ Thread suspend/resume demo completed!



## APIs Used

| API | Description |
|-----|-------------|
| `FoundryChatClient()` | Create Azure AI Foundry chat client |
| `Agent(client=...)` | Create agent with tools and instructions |
| `thread.serialize()` | Save thread state |
| `agent.deserialize_thread()` | Restore thread state |